In [ ]:
# ===============================================
# Software Cost Estimation: Full Pipeline
# ===============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# -----------------------------
# 1️⃣ Load CSV
# -----------------------------
csv_path = "data/processed/china.csv"
df = pd.read_csv(csv_path)
print("CSV loaded successfully! Shape:", df.shape)

# -----------------------------
# 2️⃣ Feature Engineering
# -----------------------------
# Functional Size
df['Functional_Size'] = df['Input'] + df['Output'] + df['Enquiry'] + df['File'] + df['Interface']

# Complexity ratios
df['Added_Deleted_Ratio'] = df['Added'] / (df['Deleted'] + 1)  # avoid division by zero
df['Change_Per_Duration'] = (df['Added'] + df['Changed'] + df['Deleted']) / df['Duration']

# Log-transform Effort for better modeling
df['Log_Effort'] = np.log1p(df['Effort'])

# Drop unnecessary columns
df_model = df.drop(['ID', 'Dev.Type', 'N_effort', 'Effort'], axis=1)

# -----------------------------
# 3️⃣ Select features & target
# -----------------------------
features = ['AFP', 'Added', 'Resource', 'Duration',
            'Functional_Size', 'Added_Deleted_Ratio', 'Change_Per_Duration']
target = 'Log_Effort'

X = df_model[features]
y = df_model[target]

# -----------------------------
# 4️⃣ Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------------
# 5️⃣ Train XGBoost Model
# -----------------------------
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# -----------------------------
# 6️⃣ Evaluate Model
# -----------------------------
y_pred_log = xgb_model.predict(X_test)
y_pred_effort = np.expm1(y_pred_log)   # convert back from log
y_test_effort = np.expm1(y_test)

mse = mean_squared_error(y_test_effort, y_pred_effort)
r2 = r2_score(y_test_effort, y_pred_effort)

print("XGBoost Performance:")
print("MSE:", mse)
print("R² Score:", r2)

# -----------------------------
# 7️⃣ Convert Effort → Cost
# -----------------------------
average_monthly_salary = 50000  # INR per developer per month
predicted_cost = y_pred_effort * average_monthly_salary

print("\nPredicted Cost for test projects (first 5 rows):")
print(predicted_cost[:5])

# -----------------------------
# 8️⃣ SHAP Explanations
# -----------------------------
explainer = shap.Explainer(xgb_model, X_train)
shap_values = explainer(X_test)

# Summary plot: feature importance
shap.summary_plot(shap_values, X_test, plot_type="bar")

# -----------------------------
# 9️⃣ Single Project Demo
# -----------------------------
# Example project input
single_project = pd.DataFrame({
    'AFP': [1200],
    'Added': [400],
    'Resource': [3],
    'Duration': [6],
    'Functional_Size': [1000],
    'Added_Deleted_Ratio': [2.0],
    'Change_Per_Duration': [50.0]
})

log_effort_single = xgb_model.predict(single_project)
effort_single = np.expm1(log_effort_single)
cost_single = effort_single[0] * average_monthly_salary

print(f"\nSingle Project Prediction:")
print(f"Predicted Effort: {effort_single[0]:.0f} person-months")
print(f"Predicted Cost: ₹{cost_single:,.0f}")

# SHAP force plot for single project
shap.initjs()
shap.force_plot(explainer.expected_value, explainer(single_project).values, single_project)